# Intraday Trading Bot Notebook

This notebook is the main place to run the project. It uses real Alpaca market data for backtesting and keeps the strategy code organized underneath.

Start at the top and run each cell in order.

## 1. Load The Bot Code

In [ ]:
from pathlib import Path
import sys

# If this notebook is inside the notebooks folder, the project root is one folder up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
import numpy as np
import pandas as pd

from infratrading_bot.backtest import prepare_signal_frame, run_long_only_backtest
from infratrading_bot.config import BotConfig, RiskConfig
from infratrading_bot.data import normalize_ohlcv
from infratrading_bot.indicators import add_all_indicators
from infratrading_bot.risk import apply_slippage
from infratrading_bot.signals import SignalAction

STARTING_CASH = 1500
MAX_OPEN_POSITIONS = 20

config = BotConfig(
    risk=RiskConfig(
        starting_cash=STARTING_CASH,
        risk_per_trade_pct=0.01,
        max_position_pct=0.35,
        max_total_exposure_pct=0.85,
        stop_loss_pct=0.005,
        take_profit_pct=0.007,
    )
)
config


## 2. Backtest The Last 3 Months With Daily XLK-Style Technology Universe

This experiment uses Alpaca 1-minute bars, an editable XLK-style technology candidate basket, and a daily top-20 active universe rebuilt using only prior market data.


In [ ]:
# Run this once if alpaca-py is not installed in your current notebook environment.
# In Jupyter, remove the # on the next line and run the cell.
# %pip install alpaca-py


In [ ]:


import os
from getpass import getpass

# This asks for your keys without showing them on screen.
# Use your Alpaca paper trading keys for now.
if not os.getenv("ALPACA_API_KEY"):
    os.environ["ALPACA_API_KEY"] = getpass("Alpaca API key: ")
if not os.getenv("ALPACA_SECRET_KEY"):
    os.environ["ALPACA_SECRET_KEY"] = getpass("Alpaca secret key: ")

In [ ]:
from datetime import datetime, time, timedelta
from zoneinfo import ZoneInfo

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
from alpaca.data.enums import DataFeed
from alpaca.trading.client import TradingClient
from alpaca.trading.enums import AssetClass, AssetStatus

MARKET_TZ = ZoneInfo("America/New_York")

# Backtest window: about the last 6 months through now.
# Change BACKTEST_DAYS if you want a shorter/faster or longer test.
BACKTEST_DAYS = 183

end = datetime.now(MARKET_TZ)
start = end - timedelta(days=BACKTEST_DAYS)

print("Backtest data window:", start, "to", end)

In [ ]:
# Daily XLK top-20 universe experiment.
#
# Important: Alpaca gives us prices/bars, not historical XLK ETF holdings.
# So this notebook uses an editable XLK-style technology candidate basket, then
# rebuilds a top-20 active universe at the start of each trading day using only
# prior market data. This avoids lookahead and gives the bot a genuinely changing
# daily universe.

BACKTEST_UNIVERSE_NAME = "Daily XLK-style technology top 20"
UNIVERSE_LOOKBACK_DAYS = 60
DAILY_UNIVERSE_LOOKBACK_DAYS = 20
DAILY_ACTIVE_UNIVERSE_SIZE = 20
MAX_FINAL_UNIVERSE = 50
MIN_PRICE = 5
MIN_AVG_VOLUME = 500_000
MIN_AVG_DOLLAR_VOLUME = 20_000_000

# Editable XLK-style candidate basket.
# The first names are the largest/current major XLK holdings; the extra names give
# the daily universe selector enough depth to rotate into stronger technology names.
XLK_TECH_CANDIDATE_UNIVERSE = [
    "NVDA", "AAPL", "MSFT", "MU", "AVGO",
    "AMD", "INTC", "CSCO", "LRCX", "AMAT",
    "ORCL", "CRM", "IBM", "NOW", "QCOM",
    "TXN", "ADI", "KLAC", "PANW", "PLTR",
    "CDNS", "SNPS", "ADBE", "INTU", "ANET",
    "APH", "MSI", "ROP", "ADSK", "FTNT",
    "TEL", "MCHP", "DELL", "HPQ", "WDC",
    "STX", "ON", "GLW", "KEYS", "MPWR",
    "TER", "FSLR", "ENPH",
]

BASE_UNIVERSE = XLK_TECH_CANDIDATE_UNIVERSE.copy()
BROAD_SEED_UNIVERSE = XLK_TECH_CANDIDATE_UNIVERSE.copy()
EXPANDED_CANDIDATES = []
NEWS_CANDIDATES = []

trading_client = TradingClient(
    os.environ["ALPACA_API_KEY"],
    os.environ["ALPACA_SECRET_KEY"],
    paper=True,
)

assets = trading_client.get_all_assets()
tradable_symbols = {
    asset.symbol
    for asset in assets
    if asset.asset_class == AssetClass.US_EQUITY
    and asset.status == AssetStatus.ACTIVE
    and asset.tradable
}

candidate_symbols = [symbol for symbol in XLK_TECH_CANDIDATE_UNIVERSE if symbol in tradable_symbols]
missing_symbols = [symbol for symbol in XLK_TECH_CANDIDATE_UNIVERSE if symbol not in tradable_symbols]

print("Universe:", BACKTEST_UNIVERSE_NAME)
print("Tradable XLK-style technology candidates:", len(candidate_symbols))
print("Missing/untradable symbols:", missing_symbols)
candidate_symbols


In [ ]:
client = StockHistoricalDataClient(
    os.environ["ALPACA_API_KEY"],
    os.environ["ALPACA_SECRET_KEY"],
)

# This cell calculates recent liquidity stats for the candidate basket.
# The actual active universe is rebuilt daily later from the 1-minute backtest data.
universe_end = datetime.now(MARKET_TZ)
universe_start = universe_end - timedelta(days=UNIVERSE_LOOKBACK_DAYS)

universe_request = StockBarsRequest(
    symbol_or_symbols=candidate_symbols,
    timeframe=TimeFrame.Day,
    start=universe_start,
    end=universe_end,
    feed=DataFeed.IEX,
)

universe_bars = client.get_stock_bars(universe_request).df.reset_index()
universe_bars["dollar_volume"] = universe_bars["close"] * universe_bars["volume"]
universe_bars = universe_bars.sort_values(["symbol", "timestamp"])
universe_bars["return"] = universe_bars.groupby("symbol")["close"].pct_change()
universe_bars["volatility"] = universe_bars.groupby("symbol")["return"].rolling(20).std().reset_index(level=0, drop=True)

universe_stats = (
    universe_bars
    .groupby("symbol")
    .agg(
        last_close=("close", "last"),
        avg_volume=("volume", "mean"),
        avg_dollar_volume=("dollar_volume", "mean"),
        avg_volatility=("volatility", "mean"),
        bars=("close", "count"),
    )
)

liquid_candidates = universe_stats[
    (universe_stats["last_close"] >= MIN_PRICE)
    & (universe_stats["avg_volume"] >= MIN_AVG_VOLUME)
    & (universe_stats["avg_dollar_volume"] >= MIN_AVG_DOLLAR_VOLUME)
].sort_values("avg_dollar_volume", ascending=False)

# Keep a broad enough pool so the daily universe can actually change.
final_universe = [symbol for symbol in liquid_candidates.index.tolist() if symbol in candidate_symbols][:MAX_FINAL_UNIVERSE]

print("Candidate pool after liquidity filter:", len(final_universe))
print("Daily active universe size:", DAILY_ACTIVE_UNIVERSE_SIZE)
universe_stats.loc[[symbol for symbol in final_universe if symbol in universe_stats.index]].sort_values("avg_dollar_volume", ascending=False)


In [ ]:
# Derivative universe scoring is skipped for this daily XLK-universe experiment.
# The trade signal now uses stationary-point derivative logic later.
ranked_derivative_universe = pd.DataFrame(index=final_universe)
print("Using daily XLK-style top-20 universe; derivative universe pre-ranking skipped.")


In [ ]:
client = StockHistoricalDataClient(
    os.environ["ALPACA_API_KEY"],
    os.environ["ALPACA_SECRET_KEY"],
)

request = StockBarsRequest(
    symbol_or_symbols=final_universe,
    timeframe=TimeFrame.Minute,
    start=start,
    end=end,
    feed=DataFeed.IEX,  # Free Alpaca feed. Use SIP only if your account has SIP access.
)

bars = client.get_stock_bars(request)
raw_bars = bars.df.reset_index()

print("Downloaded rows:", len(raw_bars))
raw_bars.head()

In [ ]:
# Validate the universe based on whether Alpaca returned enough usable candles.
min_candles_required = 20
available_counts = raw_bars.groupby("symbol").size().sort_values(ascending=False)
valid_universe = available_counts[available_counts >= min_candles_required].index.tolist()

print("Valid symbols:", len(valid_universe))
available_counts

In [ ]:
def build_daily_xlk_top20_universe_schedule(raw_bars, valid_universe):
    """Build a daily top-20 universe from the XLK-style candidate pool.

    Ranking uses only data strictly before the trading day being ranked, so the
    backtest does not peek into the future. The score favours liquidity first,
    then recent strength and useful intraday volatility.
    """
    bars = raw_bars[raw_bars["symbol"].isin(valid_universe)].copy()
    bars["timestamp"] = pd.to_datetime(bars["timestamp"])
    if bars["timestamp"].dt.tz is None:
        bars["timestamp"] = bars["timestamp"].dt.tz_localize("UTC")
    bars["timestamp_ny"] = bars["timestamp"].dt.tz_convert(MARKET_TZ)
    bars["date"] = bars["timestamp_ny"].dt.date
    bars["dollar_volume"] = bars["close"] * bars["volume"]

    daily = (
        bars
        .sort_values(["symbol", "timestamp_ny"])
        .groupby(["symbol", "date"])
        .agg(
            close=("close", "last"),
            volume=("volume", "sum"),
            dollar_volume=("dollar_volume", "sum"),
            intraday_range=("high", lambda x: float(x.max())),
            intraday_low=("low", lambda x: float(x.min())),
            bars=("close", "count"),
        )
        .reset_index()
    )
    daily["daily_return"] = daily.groupby("symbol")["close"].pct_change()
    daily["intraday_range_pct"] = (daily["intraday_range"] - daily["intraday_low"]) / daily["close"]

    schedule = {}
    details = []
    trading_days = sorted(daily["date"].unique())
    fallback_universe = valid_universe[:DAILY_ACTIVE_UNIVERSE_SIZE]

    for day in trading_days:
        history = daily[daily["date"] < day].copy()
        if history.empty:
            selected = fallback_universe
        else:
            recent_days = sorted(history["date"].unique())[-DAILY_UNIVERSE_LOOKBACK_DAYS:]
            recent = history[history["date"].isin(recent_days)].copy()
            stats = (
                recent
                .groupby("symbol")
                .agg(
                    avg_dollar_volume=("dollar_volume", "mean"),
                    avg_volume=("volume", "mean"),
                    last_close=("close", "last"),
                    first_close=("close", "first"),
                    momentum=("daily_return", lambda x: (1 + x.dropna()).prod() - 1 if len(x.dropna()) else 0.0),
                    volatility=("daily_return", "std"),
                    avg_intraday_range_pct=("intraday_range_pct", "mean"),
                    days=("date", "nunique"),
                )
                .fillna(0)
            )
            stats = stats[
                (stats["last_close"] >= MIN_PRICE)
                & (stats["avg_volume"] >= MIN_AVG_VOLUME)
                & (stats["avg_dollar_volume"] >= MIN_AVG_DOLLAR_VOLUME)
            ].copy()
            if stats.empty:
                selected = fallback_universe
            else:
                stats["liquidity_rank"] = stats["avg_dollar_volume"].rank(pct=True)
                stats["momentum_rank"] = stats["momentum"].rank(pct=True)
                stats["range_rank"] = stats["avg_intraday_range_pct"].rank(pct=True)
                stats["daily_universe_score"] = (
                    0.60 * stats["liquidity_rank"]
                    + 0.25 * stats["momentum_rank"]
                    + 0.15 * stats["range_rank"]
                )
                selected = stats.sort_values("daily_universe_score", ascending=False).head(DAILY_ACTIVE_UNIVERSE_SIZE).index.tolist()

        schedule[day] = set(selected)
        details.append({
            "date": day,
            "active_symbols": selected,
            "active_count": len(selected),
        })

    details_df = pd.DataFrame(details).set_index("date")
    return schedule, details_df


daily_universe_schedule, daily_universe_details = build_daily_xlk_top20_universe_schedule(raw_bars, valid_universe)

print("Daily XLK universe days:", len(daily_universe_schedule))
print("Average active symbols per day:", round(daily_universe_details["active_count"].mean(), 2))
display(daily_universe_details.tail())


In [ ]:
# Broad-market regime filter using SPY and QQQ.
# This blocks new long entries during weak market conditions, but sells remain allowed.
REGIME_FILTER_ENABLED = True
REGIME_SYMBOLS = ["SPY", "QQQ"]
REGIME_RECENT_RETURN_WINDOW = 2  # 2 x 15-minute bars = about 30 minutes
SPY_MIN_RECENT_RETURN = -0.003   # -0.30%
QQQ_MIN_RECENT_RETURN = -0.004   # -0.40%
REGIME_MA_WINDOW = 12            # 12 x 15-minute bars = about 3 hours

regime_request = StockBarsRequest(
    symbol_or_symbols=REGIME_SYMBOLS,
    timeframe=TimeFrame.Minute,
    start=start,
    end=end,
    feed=DataFeed.IEX,
)

regime_raw_bars = client.get_stock_bars(regime_request).df.reset_index()
print("Downloaded regime rows:", len(regime_raw_bars))

def _prepare_regime_symbol_frame(regime_raw_bars, symbol):
    symbol_rows = regime_raw_bars[regime_raw_bars["symbol"] == symbol].copy()
    candles = normalize_ohlcv(
        symbol_rows[["timestamp", "open", "high", "low", "close", "volume"]]
    )
    candles = candles.tz_convert("America/New_York").between_time("09:30", "16:00")

    df = pd.DataFrame({
        "open": candles["open"].resample(DECISION_INTERVAL).first(),
        "high": candles["high"].resample(DECISION_INTERVAL).max(),
        "low": candles["low"].resample(DECISION_INTERVAL).min(),
        "close": candles["close"].resample(DECISION_INTERVAL).last(),
        "volume": candles["volume"].resample(DECISION_INTERVAL).sum(),
    }).dropna(subset=["open", "high", "low", "close"])

    typical_price = (df["high"] + df["low"] + df["close"]) / 3
    # Intraday VWAP should reset each trading day.
    dollar_volume = typical_price * df["volume"]
    df["vwap"] = dollar_volume.groupby(df.index.date).cumsum() / df["volume"].groupby(df.index.date).cumsum()
    df["short_ma"] = df["close"].rolling(REGIME_MA_WINDOW).mean()
    df["recent_return"] = df["close"].pct_change(REGIME_RECENT_RETURN_WINDOW)
    df["above_vwap_or_ma"] = (df["close"] >= df["vwap"]) | (df["close"] >= df["short_ma"])
    return df

def compute_market_regime_filter(regime_raw_bars):
    spy = _prepare_regime_symbol_frame(regime_raw_bars, "SPY")
    qqq = _prepare_regime_symbol_frame(regime_raw_bars, "QQQ")

    regime = pd.DataFrame(index=spy.index.union(qqq.index).sort_values())
    regime["spy_recent_return"] = spy["recent_return"].reindex(regime.index).ffill()
    regime["qqq_recent_return"] = qqq["recent_return"].reindex(regime.index).ffill()
    regime["spy_above_vwap_or_ma"] = spy["above_vwap_or_ma"].reindex(regime.index).ffill().fillna(False)
    regime["qqq_above_vwap_or_ma"] = qqq["above_vwap_or_ma"].reindex(regime.index).ffill().fillna(False)

    regime["spy_regime_ok"] = (
        regime["spy_above_vwap_or_ma"]
        & (regime["spy_recent_return"] > SPY_MIN_RECENT_RETURN)
    )
    regime["qqq_regime_ok"] = (
        regime["qqq_above_vwap_or_ma"]
        & (regime["qqq_recent_return"] > QQQ_MIN_RECENT_RETURN)
    )
    regime["market_regime_ok"] = regime["spy_regime_ok"] & regime["qqq_regime_ok"]
    return regime

market_regime = compute_market_regime_filter(regime_raw_bars) if REGIME_FILTER_ENABLED else None
if market_regime is not None:
    print("Market regime OK ratio:", round(float(market_regime["market_regime_ok"].mean()), 3))
    display(market_regime.tail())


## 3. Stationary-Point Derivative Signal

Decisions happen on 15-minute windows built from 1-minute bars. The strategy buys near stationary local minima and sells near stationary local maxima without falling back to raw second_diff sign rules.


In [ ]:
# Stationary-point derivative settings.
# These multipliers define the "leeway" around stationary points using each window's own noise level.
MIN_STATIONARY_WINDOW_SAMPLES = 10
CANDLE_MOVE_EWM_SPAN = 3
SLOPE_LEEWAY_MULTIPLIER = 0.25
CURVATURE_LEEWAY_MULTIPLIER = 0.10
SELL_CURVATURE_LEEWAY_MULTIPLIER = 0.07
FLAT_SLOPE_MULTIPLIER = 0.15
FLAT_CURVATURE_MULTIPLIER = 0.05
MIN_WINDOW_MOVE_STD = 1e-6
TREND_TOLERANCE = 0.995


def _add_daily_vwap(df):
    """Add intraday VWAP that resets at each New York trading day."""
    result = df.copy()
    if result.index.tz is None:
        session = result.index.date
    else:
        session = result.index.tz_convert("America/New_York").date
    typical_price = (result["high"] + result["low"] + result["close"]) / 3
    pv = typical_price * result["volume"]
    cum_pv = pv.groupby(session).cumsum()
    cum_vol = result["volume"].groupby(session).cumsum().replace(0, np.nan)
    result["vwap"] = cum_pv / cum_vol
    return result


def _stationary_window_features(window):
    """Return stationary-point derivative features for one decision window.

    The fitted variable is percentage candle movement:
        candle_move_pct = (close - open) / open

    We fit y = a*x^2 + b*x + c over x in [0, 1].
    first_derivative_start = b
    first_derivative_end = 2*a + b
    second_derivative_window = 2*a
    """
    window = window.dropna(subset=["smooth_candle_move"])
    n = len(window)
    if n < MIN_STATIONARY_WINDOW_SAMPLES:
        return pd.Series({
            "open": np.nan,
            "high": np.nan,
            "low": np.nan,
            "close": np.nan,
            "volume": np.nan,
            "candle_move_mean": np.nan,
            "candle_move_std": np.nan,
            "first_derivative_start": np.nan,
            "first_derivative_end": np.nan,
            "second_derivative_window": np.nan,
            "fit_r2": np.nan,
        })

    y = window["smooth_candle_move"].astype(float).to_numpy()
    x = np.linspace(0.0, 1.0, n)

    # Quadratic fit over the entire 15-minute shape.
    a, b, c = np.polyfit(x, y, deg=2)
    y_hat = a * x**2 + b * x + c
    ss_res = float(np.sum((y - y_hat) ** 2))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    fit_r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot

    return pd.Series({
        "open": float(window["open"].iloc[0]),
        "high": float(window["high"].max()),
        "low": float(window["low"].min()),
        "close": float(window["close"].iloc[-1]),
        "volume": float(window["volume"].sum()),
        "candle_move_mean": float(window["candle_move_pct"].mean()),
        "candle_move_std": float(max(window["candle_move_pct"].std(ddof=0), MIN_WINDOW_MOVE_STD)),
        "first_derivative_start": float(b),
        "first_derivative_end": float(2 * a + b),
        "second_derivative_window": float(2 * a),
        "fit_r2": float(fit_r2) if pd.notna(fit_r2) else np.nan,
    })


def prepare_turning_point_signal_frame(frame, config):
    minute_df = frame.copy().sort_index()

    # Use percentage candle movement so the signal is comparable across $20 and $900 stocks.
    minute_df["candle_move_pct"] = (minute_df["close"] - minute_df["open"]) / minute_df["open"].replace(0, np.nan)
    minute_df["smooth_candle_move"] = minute_df["candle_move_pct"].ewm(
        span=CANDLE_MOVE_EWM_SPAN,
        adjust=False,
    ).mean()

    # Build one row per 15-minute decision window using the full window shape.
    # Resampler.apply can pass one Series at a time in newer pandas versions,
    # so iterate over windows explicitly to keep the helper on full DataFrames.
    window_rows = []
    for window_start, window in minute_df.resample(DECISION_INTERVAL):
        features = _stationary_window_features(window)
        features.name = window_start
        window_rows.append(features)
    df = pd.DataFrame(window_rows)
    df = df.dropna(subset=["open", "high", "low", "close", "volume"])

    # Intraday VWAP and trend features used as safety filters.
    df = _add_daily_vwap(df)
    df["smooth_close"] = df["close"].ewm(span=3, adjust=False).mean()
    df["fast_ma"] = df["smooth_close"].rolling(config.strategy.fast_ma_window).mean()
    df["slow_ma"] = df["smooth_close"].rolling(config.strategy.slow_ma_window).mean()
    df["return"] = df["smooth_close"].pct_change()
    df["momentum_3"] = df["return"].rolling(3).mean()
    df["prev_momentum_3"] = df["momentum_3"].shift(1)
    df["volatility"] = df["return"].rolling(config.strategy.volatility_window).std()

    # Keep old names as aliases so the downstream diagnostics/backtest remain compatible.
    df["second_diff"] = df["second_derivative_window"]
    df["second_diff_pct"] = df["second_derivative_window"]
    df["acceleration"] = df["second_derivative_window"]
    df["prev_acceleration"] = df["acceleration"].shift(1)
    df["prev_return"] = df["return"].shift(1)

    # Volatility-adjusted leeway around stationary points.
    df["slope_leeway"] = SLOPE_LEEWAY_MULTIPLIER * df["candle_move_std"]
    df["curvature_leeway"] = CURVATURE_LEEWAY_MULTIPLIER * df["candle_move_std"]
    df["flat_slope_leeway"] = FLAT_SLOPE_MULTIPLIER * df["candle_move_std"]
    df["flat_curvature_leeway"] = FLAT_CURVATURE_MULTIPLIER * df["candle_move_std"]

    required = [
        "close",
        "volume",
        "return",
        "volatility",
        "vwap",
        "slow_ma",
        "first_derivative_start",
        "first_derivative_end",
        "second_derivative_window",
        "slope_leeway",
        "curvature_leeway",
    ]
    has_history = ~df[required].isna().any(axis=1)
    liquid = (df["close"] >= config.strategy.min_price) & (df["volume"] >= config.strategy.min_candle_volume)
    volatility_ok = df["volatility"].between(config.strategy.min_volatility, config.strategy.max_volatility)
    trend_ok = (df["close"] >= df["slow_ma"] * TREND_TOLERANCE) & (df["close"] >= df["vwap"] * TREND_TOLERANCE)

    # Local minimum candidate: the window starts with falling movement, ends near stationary,
    # and the full-window curvature is meaningfully positive.
    df["near_stationary_min"] = df["first_derivative_end"].abs() <= df["slope_leeway"]
    df["came_from_falling"] = df["first_derivative_start"] < -df["slope_leeway"]
    df["curving_up"] = df["second_derivative_window"] > df["curvature_leeway"]

    # Local maximum candidate: the window starts with rising movement, then the end-slope
    # is near zero or has begun turning negative. This is intentionally looser than the
    # buy-side stationary test so profitable positions have more chances to exit naturally.
    df["sell_curvature_leeway"] = SELL_CURVATURE_LEEWAY_MULTIPLIER * df["candle_move_std"]
    df["near_or_negative_stationary_max"] = df["first_derivative_end"] <= df["slope_leeway"]
    df["came_from_rising"] = df["first_derivative_start"] > df["flat_slope_leeway"]
    df["curving_down"] = df["second_derivative_window"] < -df["sell_curvature_leeway"]

    df["stationary_min_candidate"] = (
        df["near_stationary_min"]
        & df["came_from_falling"]
        & df["curving_up"]
    )
    df["stationary_max_candidate"] = (
        df["near_or_negative_stationary_max"]
        & df["came_from_rising"]
        & df["curving_down"]
    )

    # Flat/stale window: both slope and curvature are too small to give useful direction.
    df["is_flat_window"] = (
        df["first_derivative_end"].abs() <= df["flat_slope_leeway"]
    ) & (
        df["second_derivative_window"].abs() <= df["flat_curvature_leeway"]
    )

    buy = (
        has_history
        & liquid
        & volatility_ok
        & trend_ok
        & df["stationary_min_candidate"]
        & ~df["is_flat_window"]
    )

    sell = has_history & df["stationary_max_candidate"]

    df["signal"] = "hold"
    df.loc[buy, "signal"] = SignalAction.BUY.value
    df.loc[sell, "signal"] = SignalAction.SELL.value

    df["entry_type"] = "none"
    df.loc[buy, "entry_type"] = "stationary_minimum_entry"

    # Rank buys by strong positive curvature and closeness to stationary slope.
    curvature_strength = df["second_derivative_window"] / df["curvature_leeway"].replace(0, np.nan)
    slope_penalty = df["first_derivative_end"].abs() / df["slope_leeway"].replace(0, np.nan)
    df["signal_score"] = (curvature_strength - slope_penalty).replace([np.inf, -np.inf], np.nan).fillna(0)

    df["signal_reason"] = "no stationary-point setup"
    df.loc[buy, "signal_reason"] = "stationary local-minimum candidate"
    df.loc[sell, "signal_reason"] = "stationary local-maximum candidate"
    df.loc[df["is_flat_window"], "signal_reason"] = "flat/stale window"
    return df


## 4. Multi-Stock Portfolio Backtest

One shared account scans the daily active universe, blocks new entries when the SPY/QQQ regime filter is weak, exits positions before the close, uses fractional sizing, and records explicit exit reasons.


In [ ]:
# Close positions before the market closes so the backtest does not carry overnight gap risk.
EOD_EXIT_TIME = time(15, 45)
LAST_ENTRY_TIME = time(14, 0)


def run_universe_portfolio_backtest(raw_bars, valid_universe, config, market_regime=None, daily_universe_schedule=None):
    signal_frames = {}
    for symbol in valid_universe:
        symbol_rows = raw_bars[raw_bars["symbol"] == symbol].copy()
        candles = normalize_ohlcv(
            symbol_rows[["timestamp", "open", "high", "low", "close", "volume"]]
        )
        candles = candles.tz_convert("America/New_York").between_time("09:30", "16:00")
        if len(candles) >= min_candles_required:
            signal_frames[symbol] = prepare_turning_point_signal_frame(candles, config)

    timeline = sorted(set().union(*[set(frame.index) for frame in signal_frames.values()]))
    cash = config.risk.starting_cash
    positions = {}
    trades = []
    entries_by_day = {}
    equity_points = []
    peak_equity = config.risk.starting_cash
    absolute_peak_equity = config.risk.starting_cash
    trailing_kill_switch_pct = 0.03
    final_cash_out_pct = 0.10
    kill_switch_events = []
    final_cash_out_event = None
    excluded_until_day = {}
    regime_blocked_buy_signals = 0
    regime_blocked_by_day = {}
    buy_signal_candidates = 0
    executed_buy_count = 0
    flat_profit_exit_count = 0

    for i in range(len(timeline) - 1):
        now = timeline[i]
        next_time = timeline[i + 1]
        current_day = now.date()
        next_is_same_day = next_time.date() == current_day
        next_clock = next_time.time()
        should_close_before_market_close = next_is_same_day and next_clock >= EOD_EXIT_TIME
        allow_new_entries = next_is_same_day and next_clock < LAST_ENTRY_TIME

        # Rebuild the tradable universe each day.
        # If daily_universe_schedule is provided, only the day's selected XLK top-20
        # symbols are eligible for NEW buys. Existing positions are still managed
        # and can still sell normally.
        if daily_universe_schedule is None:
            todays_base_universe = set(signal_frames.keys())
        else:
            todays_base_universe = set(daily_universe_schedule.get(current_day, set(valid_universe)))

        active_symbols = {
            symbol
            for symbol in todays_base_universe
            if symbol in signal_frames and excluded_until_day.get(symbol, current_day) <= current_day
        }

        # Mark open positions to market using the latest close available at this timestamp.
        market_value = 0.0
        for symbol, position in positions.items():
            frame = signal_frames[symbol]
            if now in frame.index:
                position["last_price"] = float(frame.loc[now, "close"])
            market_value += position["shares"] * position["last_price"]
        equity = cash + market_value
        peak_equity = max(peak_equity, equity)
        absolute_peak_equity = max(absolute_peak_equity, equity)
        equity_points.append((now, equity))

        if equity <= absolute_peak_equity * (1 - final_cash_out_pct):
            final_cash_out_event = {
                "time": now,
                "equity": equity,
                "absolute_peak_equity": absolute_peak_equity,
                "cash_out_level": absolute_peak_equity * (1 - final_cash_out_pct),
                "drawdown_pct": equity / absolute_peak_equity - 1,
            }
            for symbol in list(positions.keys()):
                frame = signal_frames[symbol]
                position = positions[symbol]
                if next_time in frame.index:
                    exit_price = apply_slippage(float(frame.loc[next_time, "open"]), "sell", config.risk.slippage_bps)
                else:
                    exit_price = apply_slippage(position["last_price"], "sell", config.risk.slippage_bps)
                pnl = (exit_price - position["entry_price"]) * position["shares"]
                cash += position["shares"] * exit_price
                trades.append({
                    "symbol": symbol,
                    "entry_time": position["entry_time"],
                    "exit_time": next_time,
                    "shares": position["shares"],
                    "entry_price": position["entry_price"],
                    "exit_price": exit_price,
                    "pnl": pnl,
                    "reason": "10% final cash-out switch",
                    "entry_type": position.get("entry_type", "unknown"),
                    "entry_second_diff": position.get("entry_second_diff"),
                    "entry_momentum_3": position.get("entry_momentum_3"),
                })
                del positions[symbol]
            equity_points.append((next_time, cash))
            break

        if positions and equity <= peak_equity * (1 - trailing_kill_switch_pct):
            killed_symbols = list(positions.keys())
            kill_switch_event = {
                "time": now,
                "equity": equity,
                "peak_equity": peak_equity,
                "drawdown_pct": equity / peak_equity - 1,
                "symbols_removed_until_next_day": killed_symbols,
            }
            kill_switch_events.append(kill_switch_event)

            for symbol in killed_symbols:
                frame = signal_frames[symbol]
                position = positions[symbol]
                if next_time in frame.index:
                    exit_price = apply_slippage(float(frame.loc[next_time, "open"]), "sell", config.risk.slippage_bps)
                else:
                    exit_price = apply_slippage(position["last_price"], "sell", config.risk.slippage_bps)
                pnl = (exit_price - position["entry_price"]) * position["shares"]
                cash += position["shares"] * exit_price
                trades.append({
                    "symbol": symbol,
                    "entry_time": position["entry_time"],
                    "exit_time": next_time,
                    "shares": position["shares"],
                    "entry_price": position["entry_price"],
                    "exit_price": exit_price,
                    "pnl": pnl,
                    "reason": "3% trailing portfolio kill switch",
                    "entry_type": position.get("entry_type", "unknown"),
                    "entry_second_diff": position.get("entry_second_diff"),
                    "entry_momentum_3": position.get("entry_momentum_3"),
                })
                # Treat this as rebuilding the universe: remove the problem symbols until tomorrow.
                excluded_until_day[symbol] = current_day + timedelta(days=1)
                del positions[symbol]

            equity_points.append((next_time, cash))
            peak_equity = cash
            continue

        # Sell first so cash is available for new buys.
        for symbol in list(positions.keys()):
            frame = signal_frames[symbol]
            if now not in frame.index or next_time not in frame.index:
                continue
            row = frame.loc[now]
            next_row = frame.loc[next_time]
            position = positions[symbol]
            next_open = float(next_row["open"])
            stop_price = position["entry_price"] * (1 - config.risk.stop_loss_pct)
            take_profit_price = position["entry_price"] * (1 + config.risk.take_profit_pct)
            # Track flat/stale windows while a position is open. If a stock stays flat,
            # the derivative signal becomes less useful. Sell profitable flat positions rather
            # than waiting for overnight exit.
            if bool(row.get("is_flat_window", False)):
                position["flat_window_count"] = position.get("flat_window_count", 0) + 1
            else:
                position["flat_window_count"] = 0

            unrealized_pnl_at_next_open = (next_open - position["entry_price"]) * position["shares"]

            if should_close_before_market_close:
                exit_reason = "end_of_day_exit"
            elif next_time.date() != now.date():
                exit_reason = "overnight_exit"
            elif next_open <= stop_price:
                exit_reason = "stop_loss"
            elif next_open >= take_profit_price:
                exit_reason = "take_profit"
            elif row["signal"] == SignalAction.SELL.value:
                exit_reason = "stationary_max_exit"
            elif position.get("flat_window_count", 0) >= 2 and unrealized_pnl_at_next_open > 0:
                exit_reason = "flat_profit_exit"
                flat_profit_exit_count += 1
            else:
                exit_reason = None

            if exit_reason:
                exit_price = apply_slippage(next_open, "sell", config.risk.slippage_bps)
                pnl = (exit_price - position["entry_price"]) * position["shares"]
                cash += position["shares"] * exit_price
                trades.append({
                    "symbol": symbol,
                    "entry_time": position["entry_time"],
                    "exit_time": next_time,
                    "shares": position["shares"],
                    "entry_price": position["entry_price"],
                    "exit_price": exit_price,
                    "pnl": pnl,
                    "reason": exit_reason,
                    "entry_type": position.get("entry_type", "unknown"),
                    "entry_second_diff": position.get("entry_second_diff"),
                    "entry_momentum_3": position.get("entry_momentum_3"),
                    "exit_first_derivative_end": float(row.get("first_derivative_end", 0)),
                    "exit_second_derivative_window": float(row.get("second_derivative_window", 0)),
                    "exit_second_diff": float(row.get("second_diff", 0)),
                    "exit_momentum_3": float(row.get("momentum_3", 0)),
                    "exit_flat_window_count": position.get("flat_window_count", 0),
                })
                del positions[symbol]

        equity = cash + sum(pos["shares"] * pos["last_price"] for pos in positions.values())
        exposure = sum(pos["shares"] * pos["last_price"] for pos in positions.values())
        max_exposure = equity * config.risk.max_total_exposure_pct

        # Buy strongest signals across the active universe.
        # The market regime filter blocks NEW long entries only. Sells above still run normally.
        if market_regime is None or now not in market_regime.index:
            market_regime_ok = True
        else:
            market_regime_ok = bool(market_regime.loc[now, "market_regime_ok"])

        candidates = []
        for symbol in active_symbols:
            frame = signal_frames[symbol]
            if symbol in positions or now not in frame.index or next_time not in frame.index:
                continue
            if not allow_new_entries:
                continue
            row = frame.loc[now]
            if row["signal"] == SignalAction.BUY.value:
                buy_signal_candidates += 1
                if market_regime_ok:
                    candidates.append((float(row["signal_score"]), symbol))
                else:
                    regime_blocked_buy_signals += 1
                    regime_blocked_by_day[current_day] = regime_blocked_by_day.get(current_day, 0) + 1

        entries_today = entries_by_day.get(current_day, 0)
        for _, symbol in sorted(candidates, reverse=True):
            if entries_today >= config.risk.max_trades_per_day or len(positions) >= MAX_OPEN_POSITIONS:
                break
            frame = signal_frames[symbol]
            next_open = float(frame.loc[next_time, "open"])
            buy_price = apply_slippage(next_open, "buy", config.risk.slippage_bps)
            available_exposure = max_exposure - exposure
            max_position_value = min(equity * config.risk.max_position_pct, available_exposure, cash)
            stop_distance = buy_price * config.risk.stop_loss_pct
            risk_budget = equity * config.risk.risk_per_trade_pct
            shares = np.floor(min(max_position_value / buy_price, risk_budget / stop_distance) * 1_000_000) / 1_000_000
            if shares <= 0:
                continue
            cash -= shares * buy_price
            exposure += shares * buy_price
            entries_today += 1
            entries_by_day[current_day] = entries_today
            entry_row = frame.loc[now]
            executed_buy_count += 1
            positions[symbol] = {
                "shares": shares,
                "entry_price": buy_price,
                "entry_time": next_time,
                "last_price": buy_price,
                "entry_type": entry_row.get("entry_type", "unknown"),
                "entry_first_derivative_end": float(entry_row.get("first_derivative_end", 0)),
                "entry_second_derivative_window": float(entry_row.get("second_derivative_window", 0)),
                "entry_second_diff": float(entry_row.get("second_diff", 0)),
                "entry_second_diff_pct": float(entry_row.get("second_diff_pct", 0)),
                "entry_momentum_3": float(entry_row.get("momentum_3", 0)),
                "entry_signal_score": float(entry_row.get("signal_score", 0)),
                "flat_window_count": 0,
            }

    # Close anything still open at the final available close.
    final_time = timeline[-1]
    for symbol, position in list(positions.items()):
        frame = signal_frames[symbol]
        final_price = float(frame.iloc[-1]["close"])
        exit_price = apply_slippage(final_price, "sell", config.risk.slippage_bps)
        pnl = (exit_price - position["entry_price"]) * position["shares"]
        cash += position["shares"] * exit_price
        trades.append({
            "symbol": symbol,
            "entry_time": position["entry_time"],
            "exit_time": final_time,
            "shares": position["shares"],
            "entry_price": position["entry_price"],
            "exit_price": exit_price,
            "pnl": pnl,
            "reason": "end of backtest",
            "entry_type": position.get("entry_type", "unknown"),
            "entry_second_diff": position.get("entry_second_diff"),
            "entry_momentum_3": position.get("entry_momentum_3"),
        })

    equity_curve = pd.Series(
        [point[1] for point in equity_points] + [cash],
        index=[point[0] for point in equity_points] + [final_time],
        name="equity",
    )
    trades_df = pd.DataFrame(trades)
    regime_stats = {
        "regime_filter_enabled": market_regime is not None,
        "regime_blocked_buy_signals": regime_blocked_buy_signals,
        "regime_blocked_days": len(regime_blocked_by_day),
        "regime_blocked_by_day": regime_blocked_by_day,
        "buy_signal_candidates": buy_signal_candidates,
        "executed_buy_count": executed_buy_count,
        "flat_profit_exit_count": flat_profit_exit_count,
        "daily_universe_enabled": daily_universe_schedule is not None,
        "daily_universe_days": 0 if daily_universe_schedule is None else len(daily_universe_schedule),
        "avg_daily_active_symbols": None if daily_universe_schedule is None else float(
            sum(len(symbols) for symbols in daily_universe_schedule.values()) / max(len(daily_universe_schedule), 1)
        ),
    }
    return equity_curve, trades_df, signal_frames, kill_switch_events, final_cash_out_event, regime_stats


In [ ]:
equity_curve, trades, signal_frames, kill_switch_events, final_cash_out_event, regime_stats = run_universe_portfolio_backtest(
    raw_bars,
    valid_universe,
    config,
    market_regime=market_regime,
    daily_universe_schedule=daily_universe_schedule,
)

def calculate_max_drawdown(equity_curve):
    running_peak = equity_curve.cummax()
    drawdown = equity_curve / running_peak - 1
    return float(drawdown.min())

def detect_sudden_portfolio_dips(equity_curve, trades=None, pct_threshold=-0.03, dollar_threshold=40):
    """Find near-vertical drops between consecutive portfolio equity points.

    pct_threshold=-0.03 means a 3%+ drop from one plotted point to the next.
    dollar_threshold=40 avoids flagging tiny moves in dollar terms.
    """
    eq = equity_curve.copy().sort_index()
    dip_df = pd.DataFrame({"equity": eq})
    dip_df["previous_equity"] = dip_df["equity"].shift(1)
    dip_df["drop_dollars"] = dip_df["equity"] - dip_df["previous_equity"]
    dip_df["drop_pct"] = dip_df["equity"] / dip_df["previous_equity"] - 1
    dip_df["date"] = dip_df.index.date

    sudden = dip_df[
        (dip_df["drop_pct"] <= pct_threshold) &
        (dip_df["drop_dollars"] <= -abs(dollar_threshold))
    ].copy()

    if sudden.empty:
        return sudden

    # Attach nearby exit reason summaries so we can see whether the drop was caused
    # by stop losses, kill switches, overnight liquidation, or something else.
    if trades is not None and not trades.empty and "exit_time" in trades.columns:
        trades_copy = trades.copy()
        trades_copy["exit_time"] = pd.to_datetime(trades_copy["exit_time"])
        summaries = []
        symbols = []
        for t in sudden.index:
            nearby = trades_copy[
                (trades_copy["exit_time"] >= t - pd.Timedelta("30min")) &
                (trades_copy["exit_time"] <= t + pd.Timedelta("30min"))
            ]
            summaries.append(nearby["reason"].value_counts().to_dict())
            symbols.append(sorted(nearby["symbol"].unique().tolist()))
        sudden["nearby_exit_reasons"] = summaries
        sudden["nearby_symbols"] = symbols

    return sudden.sort_values("drop_dollars")

# Tune these if the table is too noisy or too empty.
SUDDEN_DIP_PCT_THRESHOLD = -0.03
SUDDEN_DIP_DOLLAR_THRESHOLD = 40
sudden_dip_events = detect_sudden_portfolio_dips(
    equity_curve,
    trades=trades,
    pct_threshold=SUDDEN_DIP_PCT_THRESHOLD,
    dollar_threshold=SUDDEN_DIP_DOLLAR_THRESHOLD,
)

exit_reason_counts = trades["reason"].value_counts().to_dict() if not trades.empty else {}
entry_type_counts = trades["entry_type"].value_counts().to_dict() if (not trades.empty and "entry_type" in trades.columns) else {}
entry_type_pnl = trades.groupby("entry_type")["pnl"].sum().to_dict() if (not trades.empty and "entry_type" in trades.columns) else {}
winning_trades = int((trades["pnl"] > 0).sum()) if not trades.empty else 0

portfolio_metrics = {
    "decision_interval": DECISION_INTERVAL,
    "starting_cash": STARTING_CASH,
    "ending_equity": float(equity_curve.iloc[-1]),
    "profit_dollars": float(equity_curve.iloc[-1] - STARTING_CASH),
    "total_return": float(equity_curve.iloc[-1] / STARTING_CASH - 1),
    "max_drawdown": calculate_max_drawdown(equity_curve),
    "number_of_trades": int(len(trades)),
    "winning_trades": winning_trades,
    "win_rate": None if trades.empty else float(winning_trades / len(trades)),
    "exit_reason_counts": exit_reason_counts,
    "regime_filter_enabled": regime_stats["regime_filter_enabled"],
    "regime_blocked_buy_signals": regime_stats["regime_blocked_buy_signals"],
    "regime_blocked_days": regime_stats["regime_blocked_days"],
    "buy_signal_candidates": regime_stats["buy_signal_candidates"],
    "executed_buy_count": regime_stats["executed_buy_count"],
    "flat_profit_exit_count": regime_stats.get("flat_profit_exit_count", 0),
    "entry_type_counts": entry_type_counts,
    "entry_type_pnl": entry_type_pnl,
    "daily_universe_enabled": regime_stats["daily_universe_enabled"],
    "daily_universe_days": regime_stats["daily_universe_days"],
    "avg_daily_active_symbols": regime_stats["avg_daily_active_symbols"],
    "sudden_dip_count": int(len(sudden_dip_events)),
    "sudden_dip_days": [] if sudden_dip_events.empty else sorted({str(d) for d in sudden_dip_events["date"].tolist()}),
    "worst_sudden_dip_time": None if sudden_dip_events.empty else sudden_dip_events.index[0],
    "worst_sudden_dip_dollars": None if sudden_dip_events.empty else float(sudden_dip_events.iloc[0]["drop_dollars"]),
    "worst_sudden_dip_pct": None if sudden_dip_events.empty else float(sudden_dip_events.iloc[0]["drop_pct"]),
    "kill_switch_triggered": len(kill_switch_events) > 0,
    "kill_switch_count": len(kill_switch_events),
    "first_kill_switch_time": None if not kill_switch_events else kill_switch_events[0]["time"],
    "first_kill_switch_drawdown_pct": None if not kill_switch_events else kill_switch_events[0]["drawdown_pct"],
    "final_cash_out_triggered": final_cash_out_event is not None,
    "final_cash_out_time": None if final_cash_out_event is None else final_cash_out_event["time"],
    "final_cash_out_level": None if final_cash_out_event is None else final_cash_out_event["cash_out_level"],
    "final_cash_out_drawdown_pct": None if final_cash_out_event is None else final_cash_out_event["drawdown_pct"],
}

portfolio_metrics


In [ ]:
winners = trades[trades["pnl"] > 0]["pnl"]
losers  = trades[trades["pnl"] < 0]["pnl"]
print(f"Avg winner: ${winners.mean():.2f}")
print(f"Avg loser:  ${losers.mean():.2f}")
print(f"Reward/risk: {abs(winners.mean() / losers.mean()):.2f}")

# Also break down by exit reason
print(trades.groupby("reason")["pnl"].agg(["sum","mean","count"]).sort_values("sum"))

In [ ]:
# Sudden near-vertical dip diagnostics.
# This prints the day/time of any sharp adjacent equity-curve drop.

if sudden_dip_events.empty:
    print(
        f"No sudden dips found using threshold "
        f"{SUDDEN_DIP_PCT_THRESHOLD:.1%} and ${SUDDEN_DIP_DOLLAR_THRESHOLD}."
    )
else:
    print("Sudden near-vertical portfolio dips detected:")
    display_cols = [
        "date",
        "previous_equity",
        "equity",
        "drop_dollars",
        "drop_pct",
        "nearby_exit_reasons",
        "nearby_symbols",
    ]
    display(sudden_dip_events[display_cols])

    worst_time = sudden_dip_events.index[0]
    print("\nWorst sudden dip time:", worst_time)
    print("Worst sudden dip date:", worst_time.date())

    if not trades.empty:
        nearby_trades = trades[
            (pd.to_datetime(trades["exit_time"]) >= worst_time - pd.Timedelta("1D")) &
            (pd.to_datetime(trades["exit_time"]) <= worst_time + pd.Timedelta("1D"))
        ].sort_values("exit_time")
        print("\nTrades exiting within +/- 1 day of the worst sudden dip:")
        display(nearby_trades)


In [ ]:
trades.sort_values("pnl", ascending=False) if not trades.empty else trades

In [ ]:
ax = equity_curve.plot(
    title=f"Universe Portfolio Backtest - Account Equity, Last {BACKTEST_DAYS} Days",
    figsize=(10, 4),
)
ax.set_ylabel("Account equity: cash + open positions ($)")
ax.set_xlabel("Time")

In [ ]:
# Monthly diagnostics help show whether profit was consistent or just one lucky period.
monthly_equity = equity_curve.resample("ME").last()
monthly_returns = monthly_equity.pct_change().dropna().to_frame("monthly_return")
monthly_returns["monthly_profit_dollars"] = monthly_equity.diff().dropna()

if trades.empty:
    print("No trades to analyze by month.")
    display(monthly_returns)
else:
    monthly_trades = trades.copy()
    monthly_trades["month"] = pd.to_datetime(monthly_trades["exit_time"]).dt.to_period("M")

    print("Trade PnL by month:")
    display(monthly_trades.groupby("month")["pnl"].agg(["sum", "mean", "count"]))

    print("Exit reasons by month:")
    display(pd.crosstab(monthly_trades["month"], monthly_trades["reason"]))

    if regime_stats.get("regime_blocked_by_day"):
        blocked_by_day = pd.Series(regime_stats["regime_blocked_by_day"], name="blocked_buy_signals")
        blocked_by_day.index = pd.to_datetime(blocked_by_day.index)
        blocked_by_month = blocked_by_day.groupby(blocked_by_day.index.to_period("M")).agg(["sum", "count"])
        blocked_by_month = blocked_by_month.rename(columns={"sum": "blocked_buy_signals", "count": "blocked_days"})
        print("Regime-blocked buy signals by month:")
        display(blocked_by_month)

    display(monthly_returns)